https://velog.io/@kungsboy/%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D-12-03.%EC%95%99%EC%83%81%EB%B8%94Ensemble-Boosting-%EB%AA%A8%EB%8D%B8-AdaBoost-Gradient-Boosting-%EC%98%88%EC%A0%9C-t1gl1ih1

[AdaBoost  ](https://humankind.tistory.com/17)

https://velog.io/@gyurili/ML-%EB%B6%80%EC%8A%A4%ED%8C%85-%EC%95%8C%EA%B3%A0%EB%A6%AC%EC%A6%98Boosting%EC%9D%B4%EB%9E%80-AdaBoost-Gradient-Boosting-XGBoost-LightGBM

In [ ]:
# 일부러 예제를 위한 틀린 데이터를 만드는게 더 힒듬(데이터만 2시간째)

import pandas as pd
import numpy as np

data = pd.DataFrame({
    "고객명" : ["철수", "영희", "민수", "지영", "수진", "준호", "민재"],
    "사용기간": [1, 24, 36, 2, 48, 3, 30],
    "월요금": [90, 40, 30, 80, 25, 85, 35],
    "만족도": [1, 4, 5, 2, 5, 1, 4],
    "문의횟수": [10, 1, 0, 8, 0, 9, 2],
    "이탈여부": [1, 0, 0, 1, 0, 0, 1]  # 🔥 일부러 섞음 (핵심)
})

data

# 1) 비슷한 조건인데 결과 다름 존재
# 2) 완벽한 분리 불가능
# 3) 반드시 틀리는 모델 발생


,고객명,사용기간,월요금,만족도,문의횟수,이탈여부
0,철수,1,90,1,10,1
1,영희,24,40,4,1,0
2,민수,36,30,5,0,0
3,지영,2,80,2,8,1
4,수진,48,25,5,0,0
5,준호,3,85,1,9,0
6,민재,30,35,4,2,1


In [27]:
X = data.drop(["고객명", "이탈여부"], axis=1)
y = data["이탈여부"]

names = data["고객명"].values

weights = np.ones(len(data)) / len(data)

print("=== 초기 가중치 ===")
for i in range(len(data)):
    print(names[i], weights[i])

=== 초기 가중치 ===
철수 0.14285714285714285
영희 0.14285714285714285
민수 0.14285714285714285
지영 0.14285714285714285
수진 0.14285714285714285
준호 0.14285714285714285
민재 0.14285714285714285


In [28]:
# 약한 모델 만들기
from sklearn.tree import DecisionTreeClassifier

model1 = DecisionTreeClassifier(max_depth=1, random_state=42)
model1.fit(X, y, sample_weight=weights)

pred1 = model1.predict(X)

data["pred1"] = pred1
data["wrong1"] = data["이탈여부"] != data["pred1"]

data[["고객명", "이탈여부", "pred1", "wrong1"]]

,고객명,이탈여부,pred1,wrong1
0,철수,1,1,False
1,영희,0,0,False
2,민수,0,0,False
3,지영,1,1,False
4,수진,0,0,False
5,준호,0,1,True
6,민재,1,1,False


In [30]:
#3단계: 틀린 고객 확인 (핵심)

print("=== 1차에서 틀린 고객 ===")
print(data[data["wrong1"] == True][["고객명", "이탈여부", "pred1"]])

=== 1차에서 틀린 고객 ===
  고객명  이탈여부  pred1
5  준호     0      1


In [ ]:
error = np.sum(weights * data["wrong1"]) / np.sum(weights)

alpha1 = 0.5 * np.log((1 - error) / (error + 1e-10)) # 가중치 업데이트 공식

for i in range(len(data)):
    if data.loc[i, "wrong1"]:
        weights[i] *= np.exp(alpha1)   # 틀린 데이터 ↑
    else:
        weights[i] *= np.exp(-alpha1)  # 맞은 데이터 ↓

weights = weights / np.sum(weights)

print("=== 업데이트된 가중치 ===")
for i in range(len(data)):
    print(names[i], round(weights[i], 4))

=== 업데이트된 가중치 ===
철수 0.0833
영희 0.0833
민수 0.0833
지영 0.0833
수진 0.0833
준호 0.5
민재 0.0833


In [32]:
# 두번째 모델
model2 = DecisionTreeClassifier(max_depth=1, random_state=42)
model2.fit(X, y, sample_weight=weights)

pred2 = model2.predict(X)

data["pred2"] = pred2
data["wrong2"] = data["이탈여부"] != data["pred2"]

data[["고객명", "이탈여부", "pred2", "wrong2"]]

,고객명,이탈여부,pred2,wrong2
0,철수,1,1,False
1,영희,0,0,False
2,민수,0,0,False
3,지영,1,1,False
4,수진,0,0,False
5,준호,0,0,False
6,민재,1,0,True
